In [0]:
from pyspark.sql.functions import current_timestamp, current_date

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("schema_name", "weather_bronze")
dbutils.widgets.text("volume_name", "raw/weather-batch")
dbutils.widgets.text("container", "dataweather")
dbutils.widgets.text("storage_account", "dlspl21databricks")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/"
target_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze/tables/weather-batch"
target_table = f"{catalog_name}.{schema_name}.ly_rainfall_data"

checkpoint_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze/checkpoints/weather-batch"
schema_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze/schemas/weather-batch"

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", schema_path)
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")    #schema evolution mode
      .load(volume_path)
      .selectExpr("*", "_metadata.file_name as source_filename") 
      .withColumn("ingestion_timestamp", current_timestamp())
      .withColumn("load_date", current_date())
)

(df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true") #merging schemas if needed
      .option("path", target_path)
      .trigger(availableNow=True)
      .toTable(target_table)
)